<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Missing Values in Pandas - Cheat Sheet</title>

</head>

<body>

<h1>📘 Handling Missing Values in Pandas</h1>
<p>A complete theory + code reference</p>

<!-- SECTION 1 -->
<div class="section">
<h2>1. What Are Missing Values?</h2>

<p>
Missing values occur when data is not recorded.
In Pandas, they are represented as:
</p>

<ul>
    <li>NaN</li>
    <li>None</li>
    <li>NaT (for time data)</li>
</ul>

<div class="note">
Missing values can reduce model accuracy and must be handled carefully.
</div>
</div>


<!-- SECTION 2 -->
<div class="section">
<h2>2. Detect Missing Values</h2>

<p>
Before cleaning, always inspect your dataset.
</p>

<code>
# Count missing values per column
df.isna().sum()

# Percentage missing
df.isna().mean() * 100

# Total missing values
df.isna().sum().sum()
</code>
</div>


<!-- SECTION 3 -->
<div class="section">
<h2>3. Visualize Missing Data</h2>

<p>
Visualization helps understand missing patterns.
</p>

<code>
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(df.isna(), cbar=False)
plt.show()
</code>
</div>


<!-- SECTION 4 -->
<div class="section">
<h2>4. Remove Missing Values</h2>

<p>
Use when only a small amount of data is missing.
</p>

<code>
# Drop rows
df.dropna(inplace=True)

# Drop columns
df.dropna(axis=1, inplace=True)

# Drop rows with too many NaNs
df.dropna(thresh=3, inplace=True)
</code>

<div class="warning">
Avoid dropna() if you will lose important data.
</div>
</div>


<!-- SECTION 5 -->
<div class="section">
<h2>5. Fill Missing Values (Imputation)</h2>

<p>
Filling is the most common method.
</p>

<h3>A. Fill With Constants</h3>

<code>
df.fillna(0, inplace=True)
df.fillna("Unknown", inplace=True)
</code>

<h3>B. Statistical Filling</h3>

<code>
# Mean
df['Salary'].fillna(df['Salary'].mean(), inplace=True)

# Median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Mode
df['City'].fillna(df['City'].mode()[0], inplace=True)
</code>
</div>


<!-- SECTION 6 -->
<div class="section">
<h2>6. Time-Series Filling</h2>

<p>
Used when data depends on time.
</p>

<code>
# Forward fill
df.fillna(method='ffill', inplace=True)

# Backward fill
df.fillna(method='bfill', inplace=True)

# Interpolation
df['Price'] = df['Price'].interpolate()
</code>
</div>


<!-- SECTION 7 -->
<div class="section">
<h2>7. Group-Based Filling</h2>

<p>
Fill values based on category groups.
</p>

<code>
df['Salary'] = df.groupby('Department')['Salary'] \
                 .transform(lambda x: x.fillna(x.mean()))
</code>
</div>


<!-- SECTION 8 -->
<div class="section">
<h2>8. Create Missing Value Indicators</h2>

<p>
Missing values can carry useful information.
</p>

<code>
df['Age_missing'] = df['Age'].isna().astype(int)
</code>
</div>


<!-- SECTION 9 -->
<div class="section">
<h2>9. Replace Fake Missing Values</h2>

<p>
Sometimes missing data is hidden.
</p>

<code>
import numpy as np

df.replace(['?', 'NA', 'null', '', 'None'], np.nan, inplace=True)
</code>
</div>


<!-- SECTION 10 -->
<div class="section">
<h2>10. Production-Ready Cleaning Pipeline</h2>

<p>
Use this in real projects.
</p>

<code>
import numpy as np

# Replace fake missing values
df.replace(['?', 'NA', 'null', '', 'None'], np.nan, inplace=True)

# Add missing flags
for col in df.columns:
    df[col + '_missing'] = df[col].isna().astype(int)

# Fill numeric columns
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical columns
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Final check
assert df.isna().sum().sum() == 0
</code>
</div>


<!-- SECTION 11 -->
<div class="section">
<h2>11. Machine Learning Imputation</h2>

<p>
Use Scikit-Learn for model training.
</p>

<code>
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy='median')
cat_imp = SimpleImputer(strategy='most_frequent')

df[num_cols] = num_imp.fit_transform(df[num_cols])
df[cat_cols] = cat_imp.fit_transform(df[cat_cols])
</code>
</div>


<!-- SECTION 12 -->
<div class="section">
<h2>12. When NOT to Fill</h2>

<ul>
    <li>More than 50% missing → Drop column</li>
    <li>Target variable missing → Investigate</li>
    <li>Missing not random → Analyze first</li>
</ul>

<code>
df.isna().mean().sort_values(ascending=False)
</code>
</div>


<!-- SECTION 13 -->
<div class="section">
<h2>13. Golden Rules</h2>

<ul>
    <li>Never blindly drop data</li>
    <li>Never blindly fill with zero</li>
    <li>Always inspect first</li>
</ul>

<div class="note">
Good missing value handling improves ML accuracy significantly.
</div>
</div>


</body>
</html>


<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Time Series Missing Values - Cheat Sheet</title>
<body>

<h1>📈 Handling Missing Values in Time-Series Data</h1>
<p>Theory + Practical Pandas Code</p>

<!-- SECTION 1 -->
<div class="section">
<h2>1. What is Time-Series Missing Data?</h2>

<p>
Time-series data is indexed by time (date, hour, second).
Missing values appear when:
</p>

<ul>
    <li>Sensors fail</li>
    <li>Markets close</li>
    <li>Data transmission breaks</li>
    <li>System crashes</li>
</ul>

<div class="note">
Preserving time order is critical in time-series cleaning.
</div>
</div>


<!-- SECTION 2 -->
<div class="section">
<h2>2. Set DateTime Index (First Step)</h2>

<p>
Always convert your time column into DateTime.
</p>

<code>
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
</code>
</div>


<!-- SECTION 3 -->
<div class="section">
<h2>3. Detect Missing Timestamps</h2>

<p>
Check if time steps are missing.
</p>

<code>
# Check frequency
df.index.inferred_freq

# Create full timeline
full_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq='D'
)

# Reindex
df = df.reindex(full_range)
</code>
</div>


<!-- SECTION 4 -->
<div class="section">
<h2>4. Basic Missing Value Detection</h2>

<code>
# Count missing values
df.isna().sum()

# Percentage missing
df.isna().mean() * 100
</code>
</div>


<!-- SECTION 5 -->
<div class="section">
<h2>5. Forward & Backward Fill</h2>

<p>
Uses previous or next known values.
</p>

<h3>Forward Fill</h3>

<code>
df.fillna(method='ffill', inplace=True)
</code>

<h3>Backward Fill</h3>

<code>
df.fillna(method='bfill', inplace=True)
</code>

<div class="note">
Best for stock prices, weather, sensor data.
</div>
</div>


<!-- SECTION 6 -->
<div class="section">
<h2>6. Linear & Polynomial Interpolation</h2>

<p>
Estimates missing values using trends.
</p>

<h3>Linear</h3>

<code>
df['value'] = df['value'].interpolate()
</code>

<h3>Time-Based</h3>

<code>
df['value'] = df['value'].interpolate(method='time')
</code>

<h3>Polynomial</h3>

<code>
df['value'] = df['value'].interpolate(method='polynomial', order=2)
</code>
</div>


<!-- SECTION 7 -->
<div class="section">
<h2>7. Rolling Window Imputation</h2>

<p>
Uses moving averages.
</p>

<code>
df['value'] = df['value'].fillna(
    df['value'].rolling(window=7, min_periods=1).mean()
)
</code>
</div>


<!-- SECTION 8 -->
<div class="section">
<h2>8. Seasonal / Lag-Based Filling</h2>

<p>
Uses past patterns.
</p>

<code>
# Fill using last week value
df['value'] = df['value'].fillna(df['value'].shift(7))

# Fill using last year
df['value'] = df['value'].fillna(df['value'].shift(365))
</code>
</div>


<!-- SECTION 9 -->
<div class="section">
<h2>9. Mark Missing Periods</h2>

<p>
Create indicators for ML.
</p>

<code>
df['missing_flag'] = df['value'].isna().astype(int)
</code>
</div>


<!-- SECTION 10 -->
<div class="section">
<h2>10. Drop Large Gaps</h2>

<p>
Remove unreliable long gaps.
</p>

<code>
# Drop rows if all values missing
df.dropna(how='all', inplace=True)

# Drop columns if too many missing
df.dropna(axis=1, thresh=0.7*len(df), inplace=True)
</code>
</div>


<!-- SECTION 11 -->
<div class="section">
<h2>11. Time-Series Cleaning Pipeline</h2>

<p>
Copy-paste for projects.
</p>

<code>
# Ensure datetime index
df.index = pd.to_datetime(df.index)

# Reindex full timeline
full_idx = pd.date_range(
    df.index.min(),
    df.index.max(),
    freq='D'
)
df = df.reindex(full_idx)

# Add missing flag
df['missing_flag'] = df['value'].isna().astype(int)

# Interpolate
df['value'] = df['value'].interpolate(method='time')

# Forward/Backward fill
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)

# Final check
assert df.isna().sum().sum() == 0
</code>
</div>


<!-- SECTION 12 -->
<div class="section">
<h2>12. When NOT to Interpolate</h2>

<ul>
    <li>Huge missing blocks</li>
    <li>Irregular events</li>
    <li>Financial crashes</li>
    <li>System failures</li>
</ul>

<div class="warning">
Interpolation can hide real anomalies.
</div>
</div>


<!-- SECTION 13 -->
<div class="section">
<h2>13. Best Practices</h2>

<ul>
    <li>Always keep time order</li>
    <li>Reindex before filling</li>
    <li>Prefer interpolation + ffill</li>
    <li>Save original data</li>
</ul>

<div class="note">
Good time-series imputation improves forecasting accuracy.
</div>
</div>


</body>
</html>
